In [1]:
import os
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

True

In [2]:
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

In [3]:
llm.invoke("What is the capital of South Korea?").content

'Seoul.'

In [4]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
)

In [5]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's KRW worth in USD right now?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'I don’t have real-time market data, so I can’t give you the exact current KRW/USD rate.\n\nWays you can get the exact rate now:\n- Quick check: search “KRW to USD” on Google, or use sites like XE, OANDA, Yahoo Finance, or Investing.com.\n- If you tell me the current rate you see (KRW per USD) or the amount in KRW you want to convert, I’ll compute the USD value for you.\n\nFor a rough ballpark, in recent times 1 USD has been around 1,300–1,400 KRW. So:\n- 1,000,000 KRW ≈ 714–769 USD (rough estimate, depends on the exact rate).\n\nWould you like me to convert a specific amount using a rate you provide? Or tell me the amount in KRW and I’ll give you a rough estimate using a typical rate.'}]


In [7]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [8]:
bfmsgs(result)

I don’t have real-time market data, so I can’t give you the exact current KRW/USD rate.

Ways you can get the exact rate now:
- Quick check: search “KRW to USD” on Google, or use sites like XE, OANDA, Yahoo Finance, or Investing.com.
- If you tell me the current rate you see (KRW per USD) or the amount in KRW you want to convert, I’ll compute the USD value for you.

For a rough ballpark, in recent times 1 USD has been around 1,300–1,400 KRW. So:
- 1,000,000 KRW ≈ 714–769 USD (rough estimate, depends on the exact rate).

Would you like me to convert a specific amount using a rate you provide? Or tell me the amount in KRW and I’ll give you a rough estimate using a typical rate.

In [9]:
def get_fx(currency: str) -> str:
    """Get exchange rate for a given currency."""
    if currency.upper() in ["KRW", "USD"]:
        return "1500"

In [10]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [11]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's KRW worth in USD right now?"}]}
)
bfmsgs(result)

Based on the latest quote I have:

- 1 USD ≈ 1500 KRW
- Therefore 1 KRW ≈ 0.0006667 USD

If you want to convert a specific amount (e.g., 10,000 KRW) to USD, tell me the amount and I’ll calculate it. Note: real-time rates vary by provider and moment.

In [12]:
# @tool # Uncomment this line to register the function as a LangChain tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL and extract text from div id='wrapper'."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"

    text = raw.decode("utf-8", errors="replace")
    soup = BeautifulSoup(text, "html.parser")
    wrapper = soup.find("div", id="wrapper")

    if wrapper:
        return wrapper.get_text(strip=True)
    return "Wrapper div not found"

In [ ]:
fetch_text_from_url("https://www.hanyang.ac.kr/re12")